# 04 — Evaluation & Model Comparison

**Project:** AI-Based Cybersecurity Threat Detection Using Machine Learning Techniques

Phase 3 tuned eight models (4 algorithms x 2 datasets). This phase puts them on
the **held-out test sets** — data the models have never seen — and asks how well
they really detect attacks. We compute accuracy, precision, recall, F1, AUC-ROC
and confusion matrices, draw the comparison charts, and auto-write
`reports/model_comparison_report.md`.

The logic lives in `src/models/evaluate.py`; this notebook drives it step by step.

In [1]:
# --- Setup -----------------------------------------------------------------
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

from src.utils import config as C
from src.models import evaluate as E

print("Setup complete.")

Setup complete.


## Step A — NSL-KDD test-set evaluation

Every metric is computed on the 22,544-row official NSL-KDD test set. The
"macro" metrics average over all five classes (Normal, DoS, Probe, R2L, U2R)
so the rare classes carry the same weight as Normal.

In [2]:
# Score all four models on the NSL-KDD held-out test set.
nsl_df, nsl_results = E.evaluate_dataset("nslkdd")

nsl_df[["model_label", "accuracy", "precision_macro", "recall_macro",
        "f1_macro", "auc_roc_macro", "test_latency_ms_per_row"]]

  evaluating nslkdd / logistic ...


  evaluating nslkdd / decision_tree ...


  evaluating nslkdd / random_forest ...


  evaluating nslkdd / xgboost ...


,model_label,accuracy,precision_macro,recall_macro,f1_macro,auc_roc_macro,test_latency_ms_per_row
0,Logistic Regression,0.7628,0.6707,0.6295,0.5572,0.9057,0.0008
1,Decision Tree,0.8163,0.7545,0.5836,0.6170,0.7672,0.0006
2,Random Forest,0.7488,0.7972,0.5081,0.5384,0.9489,0.0097
3,XGBoost,0.7782,0.8281,0.5653,0.6086,0.9491,0.0027


## Step B — CICIDS2017 test-set evaluation

The CICIDS2017 test set is our own stratified 20% hold-out (15,606 rows),
which keeps every class — including Heartbleed's 2 test rows.

In [3]:
# Score all four models on the CICIDS2017 held-out test set.
cic_df, cic_results = E.evaluate_dataset("cicids")

cic_df[["model_label", "accuracy", "precision_macro", "recall_macro",
        "f1_macro", "auc_roc_macro", "test_latency_ms_per_row"]]

  evaluating cicids / logistic ...


  evaluating cicids / decision_tree ...


  evaluating cicids / random_forest ...


  evaluating cicids / xgboost ...


,model_label,accuracy,precision_macro,recall_macro,f1_macro,auc_roc_macro,test_latency_ms_per_row
0,Logistic Regression,0.9549,0.8391,0.8622,0.8267,0.9784,0.0007
1,Decision Tree,0.9972,0.9951,0.9257,0.9505,0.9629,0.0005
2,Random Forest,0.9970,0.9925,0.8944,0.9276,0.9919,0.0063
3,XGBoost,0.9988,0.9974,0.9832,0.9897,1.0000,0.0067


## Step C — Comparison charts

Three kinds of chart are saved per dataset (also copied into `frontend/public/`
for the web app):

1. **Bar chart** — the headline metrics side by side.
2. **ROC curves overlaid** — each model's attack-detection trade-off curve.
3. **Confusion matrix heatmaps** — which classes each model confuses.

In [4]:
# Draw and save all charts for both datasets.
for ds, df, results in [("nslkdd", nsl_df, nsl_results),
                        ("cicids", cic_df, cic_results)]:
    E.plot_metric_bars(df, ds)
    E.plot_roc_overlay(ds, results)
    E.plot_confusion_grid(ds, results)
    print(f"charts saved for {ds}")

import glob
print("\nFiles in reports/figures/:")
for p in sorted(glob.glob(str(C.REPORTS_DIR / "figures" / "*.png"))):
    print("  -", Path(p).name)

charts saved for nslkdd


charts saved for cicids

Files in reports/figures/:
  - bar_metrics_cicids.png
  - bar_metrics_nslkdd.png
  - cicids2017_class_distribution.png
  - cicids2017_correlation.png
  - confusion_cicids.png
  - confusion_nslkdd.png
  - feature_selection_comparison.png
  - nsl_kdd_class_distribution.png
  - nsl_kdd_correlation.png
  - roc_cicids.png
  - roc_nslkdd.png


## Step D — Evaluation results JSON + the comparison report

The metrics are exported as JSON (the `/compare` API endpoint and the frontend
read this) and the written report is auto-generated into
`reports/model_comparison_report.md`.

In [5]:
# Save JSON (data/processed + frontend/public) and generate the report.
E.run_all()
print("\nEvaluation results JSON and report generated.")

=== Evaluating nslkdd ===
  evaluating nslkdd / logistic ...


  evaluating nslkdd / decision_tree ...


  evaluating nslkdd / random_forest ...


  evaluating nslkdd / xgboost ...


        model  accuracy  precision_macro  recall_macro  f1_macro  auc_roc_macro
     logistic    0.7628           0.6707        0.6295    0.5572         0.9057
decision_tree    0.8163           0.7545        0.5836    0.6170         0.7672
random_forest    0.7488           0.7972        0.5081    0.5384         0.9489
      xgboost    0.7782           0.8281        0.5653    0.6086         0.9491
=== Evaluating cicids ===
  evaluating cicids / logistic ...


  evaluating cicids / decision_tree ...


  evaluating cicids / random_forest ...


  evaluating cicids / xgboost ...


        model  accuracy  precision_macro  recall_macro  f1_macro  auc_roc_macro
     logistic    0.9549           0.8391        0.8622    0.8267         0.9784
decision_tree    0.9972           0.9951        0.9257    0.9505         0.9629
random_forest    0.9970           0.9925        0.8944    0.9276         0.9919
      xgboost    0.9988           0.9974        0.9832    0.9897         1.0000

Report written: D:\Cyber threat Detection\reports\model_comparison_report.md

Evaluation results JSON and report generated.


## Summary

* All eight models scored on untouched test sets, both datasets separately.
* Comparison charts (bars / ROC / confusion matrices) saved to `reports/figures/`
  and mirrored into `frontend/public/`.
* `reports/model_comparison_report.md` auto-generated with a results table,
  per-class recall, and interpretation.
* Phase 5 will wrap these models behind a FastAPI backend.